In [0]:
%run ../00-common/config

In [0]:
%run ./00_silver_helpers

In [0]:
from pyspark.sql import functions as F

orders = spark.table(f"{catalog_name}.{bronze_schema}.orders").drop("ingestion_timestamp", "source_file")

orders_silver = (
    orders
    # funnel columns,, kohet ne ore/dite mes fazave
    .withColumn("approval_hours",
        (F.col("order_approved_at").cast("long") - F.col("order_purchase_timestamp").cast("long")) / 3600)
    .withColumn("handling_hours",
        (F.col("order_delivered_carrier_date").cast("long") - F.col("order_approved_at").cast("long")) / 3600)
    .withColumn("shipping_days",
        (F.col("order_delivered_customer_date").cast("long") - F.col("order_delivered_carrier_date").cast("long")) / 86400)
    .withColumn("total_delivery_days",
        (F.col("order_delivered_customer_date").cast("long") - F.col("order_purchase_timestamp").cast("long")) / 86400)
    # variance: sa dite para/pas parashikimit (negativ = para kohe, pozitiv = vonuar)
    .withColumn("delivery_variance_days",
        (F.col("order_delivered_customer_date").cast("long") - F.col("order_estimated_delivery_date").cast("long")) / 86400)
)
write_to_silver(orders_silver, "orders", catalog_name, silver_schema)

In [0]:
spark.sql("SHOW TABLES IN olist.silver").show()